# Z3-Python-09 : L'enigme d'Einstein (Zebra puzzle)

[← Z3-Python-08 (ordonnancement)](Z3-08-Ordonnancement-Python.ipynb) | [Serie Z3-Python](README.md) | [Version C#](Z3-09-Enigme-Einstein-CSharp.ipynb) | [Z3-Python-10 (cryptarithmes) →](Z3-10-Cryptarithmetic-Python.ipynb)

## Le problème

L'**enigme d'Einstein** (ou *Zebra puzzle*) est un classique de logique : 5 maisons alignees, chacune habitee par une personne de nationalite, couleur, boisson, cigarette et animal différents. On dispose de **15 indices** (le Britannique vit dans la rouge, le Danois boit du the, la verte est a gauche de la blanche, etc.) et la question est : **qui possede le poisson ?**

A la main, la resolution procede par deductions progressives sur un tableau - c'est long et trompeur. L'espace de recherche brut est enorme : chaque attribut est une permutation des 5 maisons, soit $(5!)^5 = 24\,883\,200\,000$ combinaisons. Une **enumeration exhaustive** (brute force) testant les 15 indices pour chaque combinaison prend plusieurs minutes.

C'est typiquement la ou un solveur SMT comme Z3 **fait la différence** : on decrit les 25 variables (position de chaque valeur), le domaine, les 5 contraintes de permutation (`Distinct`) et les 15 indices, et `Solver` trouve l'unique solution en **quelques millisecondes**. Ce notebook porte en Python l'exemple C# [18_Einsteins_Riddle](../Z3-Linq2Z3/18_Einsteins_Riddle.ipynb) de la serie sœur Z3.Linq.

In [1]:
# Imports : z3 (solveur SMT) + time (benchmark).
import time
import z3

print("z3 version :", z3.get_version_string())

z3 version : 5.4.1


## Modelisation : l'encodage par position

L'astuce centrale consiste a **inverser le point de vue**. Au lieu de demander « quelle couleur a la maison *k* ? », on définit pour chaque valeur d'attribut une variable entiere egale a la **position** (numéro de maison 0..4) ou elle se trouve :

- `Brit`, `Swede`, `Dane`, `Norwegian`, `German` : position de chaque nationalite
- `Red`, `Green`, `White`, `Yellow`, `Blue` : position de chaque couleur
- `Tea`, `Coffee`, `Milk`, `Water`, `Beer` : position de chaque boisson
- `PallMall`, `Dunhill`, `Blend`, `BlueMaster`, `Prince` : position de chaque cigarette
- `Dog`, `Bird`, `Cat`, `Horse`, `Fish` : position de chaque animal

Soit **25 variables entieres**. Les contraintes se traduisent alors très naturellement :

1. **Domaine** : chaque variable $\in \{0,1,2,3,4\}$.
2. **Permutation** : dans chaque groupe d'attributs, les 5 positions sont deux a deux distinctes (`Distinct`) - chaque valeur occupe une maison différente.
3. **Les 15 indices** : des egalites de position (`Brit == Red` : le Britannique et la rouge sont dans la même maison) ou des **adjacences** (`|Blend - Cat| == 1` : le fumeur de Blend est voisin du Chat).

In [2]:
def build_einstein_solver():
    """Construit le solveur Z3 de l'enigme d'Einstein (25 vars de position, 15 indices).
    Retourne (solver, var_dict, group_labels) pour reuse dans les exercices."""
    s = z3.Solver()
    # 25 variables de position (0..4) : une par valeur d'attribut.
    groups = {
        "Nationalite": ["Brit", "Swede", "Dane", "Norwegian", "German"],
        "Couleur":     ["Red", "Green", "White", "Yellow", "Blue"],
        "Boisson":     ["Tea", "Coffee", "Milk", "Water", "Beer"],
        "Cigarette":   ["PallMall", "Dunhill", "Blend", "BlueMaster", "Prince"],
        "Animal":      ["Dog", "Bird", "Cat", "Horse", "Fish"],
    }
    v = {n: z3.Int(n) for grp in groups.values() for n in grp}

    # (1) Domaine 0..4 pour les 25 variables
    for var in v.values():
        s.add(var >= 0, var <= 4)

    # (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes
    for grp in groups.values():
        s.add(z3.Distinct([v[n] for n in grp]))

    # (3) Les 15 indices
    s.add(v["Brit"] == v["Red"])                                         # 1. le Britannique vit dans la rouge
    s.add(v["Swede"] == v["Dog"])                                        # 2. le Suedois eleve des chiens
    s.add(v["Dane"] == v["Tea"])                                         # 3. le Danois boit du the
    s.add(v["Green"] == v["White"] - 1)                                  # 4. la verte est juste a gauche de la blanche
    s.add(v["Green"] == v["Coffee"])                                     # 5. la verte boit du cafe
    s.add(v["PallMall"] == v["Bird"])                                    # 6. Pall Mall eleve des oiseaux
    s.add(v["Yellow"] == v["Dunhill"])                                   # 7. la jaune fume du Dunhill
    s.add(v["Milk"] == 2)                                                # 8. la maison du milieu boit du lait
    s.add(v["Norwegian"] == 0)                                           # 9. le Norvegien dans la 1ere maison
    s.add(z3.Or(v["Blend"] - v["Cat"] == 1, v["Cat"] - v["Blend"] == 1))     # 10. Blend voisin du Chat
    s.add(z3.Or(v["Horse"] - v["Dunhill"] == 1, v["Dunhill"] - v["Horse"] == 1))  # 11. Cheval voisin du Dunhill
    s.add(v["BlueMaster"] == v["Beer"])                                  # 12. BlueMaster boit de la biere
    s.add(v["German"] == v["Prince"])                                    # 13. l'Allemand fume du Prince
    s.add(z3.Or(v["Norwegian"] - v["Blue"] == 1, v["Blue"] - v["Norwegian"] == 1))  # 14. Norvegien voisin de la bleue
    s.add(z3.Or(v["Blend"] - v["Water"] == 1, v["Water"] - v["Blend"] == 1))   # 15. Blend voisin de l'Eau
    return s, v, groups

print("Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.")

Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.


## Resolution et temoin



On appelle `s.check()` : si `sat`, on extrait le modèle puis on **inverse** l'encodage par position - pour chaque maison $h \in \{0..4\}$, on cherche quelle valeur de chaque attribut y se trouve (la valeur dont la position vaut $h$). Cela reconstitue le tableau classique de la solution.

In [3]:
# Labels francais pour l'affichage (associes au nom de variable).
LABELS_FR = {
    "Nationalite": [("Britannique", "Brit"), ("Suedois", "Swede"), ("Danois", "Dane"), ("Norvegien", "Norwegian"), ("Allemand", "German")],
    "Couleur":     [("Rouge", "Red"), ("Verte", "Green"), ("Blanche", "White"), ("Jaune", "Yellow"), ("Bleue", "Blue")],
    "Boisson":     [("The", "Tea"), ("Cafe", "Coffee"), ("Lait", "Milk"), ("Eau", "Water"), ("Biere", "Beer")],
    "Cigarette":   [("Pall Mall", "PallMall"), ("Dunhill", "Dunhill"), ("Blend", "Blend"), ("BlueMaster", "BlueMaster"), ("Prince", "Prince")],
    "Animal":      [("Chien", "Dog"), ("Oiseau", "Bird"), ("Chat", "Cat"), ("Cheval", "Horse"), ("Poisson", "Fish")],
}

def solve_and_display():
    """Resout l'enigme et affiche le tableau des 5 maisons."""
    s, v, _ = build_einstein_solver()
    assert s.check() == z3.sat, "UNSAT (inattendu pour cette enigme)"
    m = s.model()
    pos = {n: m[v[n]].as_long() for n in v}  # nom de variable -> position (0..4)

    def val_at(group, h):
        """Retourne le label francais de la valeur du groupe present a la maison h."""
        for fr, key in LABELS_FR[group]:
            if pos[key] == h:
                return fr
        return "?"

    cols = ["Maison", "Nationalite", "Couleur", "Boisson", "Cigarette", "Animal"]
    widths = [7, 13, 9, 9, 13, 9]
    print("  ".join(c.ljust(w) for c, w in zip(cols, widths)))
    print("  " + "-" * (sum(widths) + 2 * (len(cols) - 1)))
    for h in range(5):
        row = [str(h + 1)] + [val_at(g, h) for g in ["Nationalite", "Couleur", "Boisson", "Cigarette", "Animal"]]
        print("  ".join(c.ljust(w) for c, w in zip(row, widths)))
    print()
    print(f">>> Reponse : c'est le {val_at('Nationalite', pos['Fish'])} qui possede le Poisson (maison {pos['Fish'] + 1}).")
    return m, v

solution_model, solution_vars = solve_and_display()

Maison   Nationalite    Couleur    Boisson    Cigarette      Animal   
  ----------------------------------------------------------------------
1        Norvegien      Jaune      Eau        Dunhill        Chat     
2        Danois         Bleue      The        Blend          Cheval   
3        Britannique    Rouge      Lait       Pall Mall      Oiseau   
4        Allemand       Verte      Cafe       Prince         Poisson  
5        Suedois        Blanche    Biere      BlueMaster     Chien    

>>> Reponse : c'est le Allemand qui possede le Poisson (maison 4).


## Interpretation

Z3 trouve l'unique solution en quelques millisecondes. La resolution humaine procederait par deductions progressives - on place le Norvegien en 1 (`Norwegian == 0`), le lait en 3 (`Milk == 2`), puis on propage les egalites et adjacences - mais c'est précisément ce travail de **propagation de contraintes** que le solveur automatise, sans erreur ni oubli.

La reponse a la question « qui possede le poisson ? » tombe directement du modèle extrait. Notons qu'**aucun indice ne parle du poisson** : c'est par elimination (les 4 autres animaux sont localises par les indices) que sa position se deduit.

## Vérification du témoin : relire les indices

Un modèle extrait n'est pas un résultat à prendre sur parole — vérifions que le tableau ci-dessus satisfait bien quelques indices choisis dans l'énoncé original :

- **Le Norvégien vit à côté de la maison bleue** : Norvégien en maison 1, Bleue en maison 2 — voisins.
- **Le fumeur de Blend a un voisin qui garde le chat** : Blend en maison 2, Chat en maison 1 — voisins.
- **La maison verte est immédiatement à gauche de la blanche** : Verte en 4, Blanche en 5.
- **La maison du centre boit du lait** : Lait en maison 3.

Ce geste de re-lecture croisée est l'équivalent d'un test unitaire sur le solveur : il valide l'**encodage** (une contrainte mal traduite produirait un témoin qui échoue à la vérification), pas seulement la satisfiabilité.

## Approche naive vs solveur : mesurer l'ecart



Plutot que d'affirmer que le solveur est plus rapide, **mesurons** l'ecart. L'espace de recherche brut est $(5!)^5$ : chaque attribut est une permutation des 5 maisons, independamment. On chronometre aussi la resolution Z3, puis on **prouve l'unicite** de la solution en ajoutant la negation du temoin trouve et en re-resolvant (on attend `unsat`).

In [4]:
# (1) Espace de recherche brut
import math
space = math.factorial(5) ** 5  # (5!)^5 = 24_883_200_000
brute_seconds = space / 1e8     # a ~1e8 tests/s, sans compter le test des 15 indices

# (2) Chronometre la resolution Z3
t0 = time.perf_counter()
s1, v1, _ = build_einstein_solver()
assert s1.check() == z3.sat
m1 = s1.model()
z3_ms = (time.perf_counter() - t0) * 1000

# (3) Unicite : on nie le temoin trouve (au moins une variable differe) -> on attend unsat
s2, v2, _ = build_einstein_solver()
negation = z3.Or([v2[n] != m1[v1[n]].as_long() for n in v2])  # une solution differente existe-t-elle ?
s2.add(negation)
unique = (s2.check() == z3.unsat)

print(f"Espace de recherche brut       : (5!)^5 = {space:,} combinaisons")
print(f"Brute force estime (~1e8/s)    : {brute_seconds:.1f} s (hors test des 15 indices)")
print(f"Z3 solve temps                 : {z3_ms:.1f} ms")
print(f"Solution unique (UNSAT si nie) : {unique}")
print(f" -> Z3 resout en millisecondes un espace de ~{space/1e9:.0f} milliards, et prouve l'unicite sans enumeration.")

Espace de recherche brut       : (5!)^5 = 24,883,200,000 combinaisons
Brute force estime (~1e8/s)    : 248.8 s (hors test des 15 indices)
Z3 solve temps                 : 10.6 ms
Solution unique (UNSAT si nie) : True
 -> Z3 resout en millisecondes un espace de ~25 milliards, et prouve l'unicite sans enumeration.


## Lecture du benchmark

Sur cette enigme, l'ecart est **frappant** : Z3 resout les 25 variables entrelacees en quelques millisecondes, alors que la brute force doit affronter **~25 milliards** de combinaisons (soit plusieurs minutes même a cadence elevee, et bien plus avec le test des 15 indices a chaque candidat).

La **propagation** fait tout le travail : `Norwegian == 0` et `Milk == 2` sont des ancres imposees par les indices 9 et 8, puis les egalites de position et les adjacences reduisent les domaines restants jusqu'a ce qu'il ne reste qu'une seule combinaison coherente. C'est exactement la mecanique qu'un humain deployerait a la main sur le tableau - mais le solveur la realise sans erreur et la **prouve complete** (unicite via `unsat` sur la negation).

## Exercices

Les trois exercices initiaux de ce carnet ont été résolus par un rendu de TP : chacun devient un **exemple guidé** (code de l'étudiante conservé tel quel), suivi d'une lecture du résultat puis d'un **nouvel exercice** non résolu, de difficulté comparable et portant le même concept Z3. Vous y réutiliserez l'encodage par position et la fonction `build_einstein_solver`.

### Exemple guidé 1 -- Un 16e indice : le Chat boit de l'Eau

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), intégré comme exemple guidé.

**Objectif** : ajouter l'indice `Cat == Water` (le Chat et l'Eau sont dans la même maison) et observer si l'énigme reste satisfaisable (`sat`) ou devient contradictoire (`unsat`).

**Rappel** : on reprend `build_einstein_solver()`, on ajoute `s.add(v["Cat"] == v["Water"])` avant `check()`, et on compare avec la solution trouvée plus haut — le Chat et l'Eau y sont-ils dans la même maison ?

In [5]:
# Exemple guidé 1 : ajout de l'indice Cat == Water et test de la satisfiabilite.
# Etape 1 : reprenez build_einstein_solver() -> s, v, _.
# Etape 2 : s.add(v["Cat"] == v["Water"]).
# Etape 3 : s.check() -> sat ou unsat ?

def tester_chat_eau():
    """Retourne 'sat' ou 'unsat' apres ajout de l'indice Cat == Water."""
    # Étape 1 : On récupère le solveur déjà configuré avec les 15 indices
    s, v, _ = build_einstein_solver()

    # Étape 2 : On ajoute le 16e indice (le Chat et l'Eau dans la même maison)
    s.add(v["Cat"] == v["Water"])

    # Étape 3 : On vérifie si le problème reste possible et on renvoie le résultat
    resultat = s.check()
    return str(resultat)  # Retournera 'sat'

# Pour tester ta fonction et afficher le résultat :
print("Verdict avec Cat == Water :", tester_chat_eau())

Verdict avec Cat == Water : sat


### Lecture du résultat : un indice déjà vrai

Le verdict est `sat` : l'énigme reste satisfaisable. Ce n'est pas un hasard — dans l'unique solution affichée plus haut, le Chat et l'Eau sont **déjà** dans la même maison (maison 1, chez le Norvégien). Le 16e indice est donc déjà vrai : il ne crée aucune contradiction et n'apporte aucune information nouvelle. Un indice qui contredirait la solution (par exemple `Fish == Water`) rendrait au contraire l'énigme `unsat` — c'est exactement l'objet du nouvel exercice 1.

### Exercice 1 -- Prédire le verdict d'un 17e indice

**Objectif** : généraliser l'exemple guidé 1 en une fonction qui classe **n'importe quel** indice candidat, puis répondre à la question conceptuelle.

Écrire `classer_indice(indice)` qui prend un indice candidat (une fonction de `v` retournant une expression booléenne Z3, sur le patron du bonus de l'exemple guidé 2), reconstruit le solveur complet, ajoute l'indice et retourne `'sat'` ou `'unsat'`. Classer ensuite les quatre candidats :

1. `lambda v: v["Cat"] == v["Water"]` — le Chat et l'Eau dans la même maison ;
2. `lambda v: v["Fish"] == v["Water"]` — le Poisson et l'Eau dans la même maison ;
3. `lambda v: v["Horse"] == v["Blend"]` — le Cheval dans la maison du fumeur de Blend ;
4. `lambda v: v["Bird"] == v["Milk"]` — l'Oiseau dans la maison au lait.

**Question** : parmi les verdicts obtenus, peut-il exister un indice qui donne `sat` tout en **changeant** la solution du tableau ? Justifier à partir de l'unicité prouvée plus haut.

**Indices** : chaque appel doit repartir d'un solveur **frais** (`build_einstein_solver()`) — sinon les indices s'accumulent d'un appel à l'autre. Pour la question : que vaut l'ensemble des solutions avant l'ajout, et que lui fait un indice déjà vrai ?

In [6]:
# Exercice 1 : classer un 17e indice candidat (sat / unsat).
# Etape 1 : une fonction classer_indice(indice) qui repart d'un solveur FRAIS a chaque appel.
# Etape 2 : classer les 4 indices candidats de l'enonce.
# Etape 3 : repondre a la question -- un indice sat peut-il changer la solution ?

def classer_indice(indice):
    """Retourne 'sat' ou 'unsat' pour l'enigme complete + l'indice candidat.
    indice : une fonction qui prend v et retourne l'expression Z3 (patron du bonus)."""
    # TODO etudiant : build_einstein_solver(), ajouter indice(v), retourner le verdict
    return None  # TODO etudiant

print("Exercice a completer : classer les 4 indices candidats.")

Exercice a completer : classer les 4 indices candidats.


### Exemple guidé 2 -- Énumérer toutes les solutions par blocage itératif

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), intégré comme exemple guidé.

**Objectif** : énumérer **toutes** les solutions de l'énigme par **blocage itératif** (on résout, on compte, on ajoute une contrainte interdisant le témoin trouvé, on re-résout) jusqu'à `unsat`. Le compte attendu est 1 (solution unique), ce qui confirme le benchmark ci-dessus. Le rendu généralise la fonction en `compter_solutions(s=None, v=None, limite=1000)` pour qu'elle s'applique à n'importe quel solveur — elle resservira dans l'exemple guidé 3 et le nouvel exercice 2.

In [7]:
# Exemple guidé 2 : énumérer toutes les solutions par blocage itératif.
# Etape 1 : s, v, _ = build_einstein_solver().
# Etape 2 : while s.check() == z3.sat: compte += 1; m = s.model(); s.add(z3.Or([v[n] != m[v[n]].as_long() for n in v])).
# Etape 3 : afficher le compte (attendu : 1).

def compter_solutions(s=None, v=None, limite=1000):
    """Retourne le nombre total de solutions (attendu : 1 pour l'enigme d'Einstein).
    Sans argument : utilise le solveur complet de l'enigme.
    On peut aussi lui passer un autre solveur (s, v), par exemple la mini-enigme de l'exemple guide 3."""
    # Etape 1 : le solveur avec les 15 indices
    if s is None:
        s, v, _ = build_einstein_solver()

    # Etape 2 : boucle de blocage iteratif
    compte = 0
    while compte < limite and s.check() == z3.sat:
        compte += 1
        m = s.model()                                            # la solution trouvee
        s.add(z3.Or([v[n] != m[v[n]].as_long() for n in v]))     # on l'interdit : au moins 1 variable doit changer
    return compte

# Etape 3 : afficher le compte
nb = compter_solutions()
print(f"Nombre de solutions de l'enigme d'Einstein : {nb} (attendu : 1)")

Nombre de solutions de l'enigme d'Einstein : 1 (attendu : 1)


In [8]:
# Bonus exemple guidé 2 : la boucle compte-t-elle vraiment ? On retire un indice et on recompte.
def solveur_sans_indice(indice):
    """Recree le solveur d'Einstein en retirant UNE contrainte (l'indice donne)."""
    s, v, _ = build_einstein_solver()
    s2 = z3.Solver()
    for a in s.assertions():
        if not z3.eq(a, indice(v)):      # on recopie toutes les contraintes sauf celle-la
            s2.add(a)
    assert len(s2.assertions()) == len(s.assertions()) - 1, "l'indice n'a pas ete trouve"
    return s2, v

tests = [
    ("9  (Norvegien en maison 1)",  lambda v: v["Norwegian"] == 0),
    ("15 (Blend voisin de l'Eau)",  lambda v: z3.Or(v["Blend"] - v["Water"] == 1, v["Water"] - v["Blend"] == 1)),
]
for nom, indice in tests:
    s2, v2 = solveur_sans_indice(indice)
    print(f"Sans l'indice {nom} : {compter_solutions(s2, v2)} solution(s)")

Sans l'indice 9  (Norvegien en maison 1) : 17 solution(s)
Sans l'indice 15 (Blend voisin de l'Eau) : 1 solution(s)


### Lecture du résultat : la boucle compte vraiment, et un indice est redondant

Le blocage itératif (trouver une solution, l'interdire par un `Or` sur les 25 variables, recommencer jusqu'à `unsat`) donne **1 solution** : l'énigme d'Einstein est bien unique, ce qui confirme le benchmark. Pour vérifier que la boucle compte réellement, on retire un indice et on recompte : sans l'indice 9 (Norvégien en maison 1), on trouve **17 solutions**. Sans l'indice 15 (Blend voisin de l'Eau), il en reste **1** : cet indice est **redondant**, les 14 autres suffisent à déterminer la solution. Le nouvel exercice 2 pousse cette question un cran plus loin.

### Exercice 2 -- Un sous-ensemble minimal d'indices

**Objectif** : sans l'indice 15, l'énigme garde exactement 1 solution — mais est-il le seul qu'on puisse retirer ? Trouver par **élimination gloutonne** un sous-ensemble des 15 indices qui garde **exactement 1 solution** : partir de la liste complète, et pour chaque indice dans l'ordre, essayer de le retirer **définitivement** — si le reste a toujours exactement 1 solution, il reste retiré, sinon il est remis. Combien d'indices le résultat conserve-t-il, et lesquels ?

**Indices** : listez d'abord les 15 indices sous forme de lambdas `lambda v: <expr>` (les deux du bonus donnent le patron) ; reconstruisez à chaque essai un solveur frais ne contenant que les indices retenus, et comptez avec `compter_solutions(s, v)`. Affichez le nombre d'indices conservés et leur liste. Attention : un indice isolément redondant peut devenir nécessaire une fois d'autres indices retirés — d'où le test à chaque étape.

In [9]:
def indices_minimaux():
    """Retourne la liste (nom, lambda) du sous-ensemble glouton qui garde exactement 1 solution."""
    # TODO etudiant : liste des 15 indices, puis boucle d'élimination gloutonne
    return None  # TODO etudiant

print("Exercice a completer : élimination gloutonne des indices redondants.")

Exercice a completer : élimination gloutonne des indices redondants.


### Exemple guidé 3 -- Mini-Zebra à 4 maisons

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026), intégré comme exemple guidé.

**Objectif** : modéliser une **mini-énigme** à 4 maisons et 3 attributs (4 nationalités, 4 couleurs, 4 animaux) avec 8 indices, et vérifier que l'encodage par position se généralise — le modèle est paramètre par les données, seule l'instance change.

In [10]:
# Exemple guidé 3 : un mini-Zebra a 4 maisons / 3 attributs.
# Indice : groups_4 = {"Nat": [...4...], "Couleur": [...4...], "Animal": [...4...]}, domaine 0..3.
# Etape 1 : définissez groups_4 (4 valeurs par attribut) et un solveur avec domaine 0..3 + 3 Distinct.
# Etape 2 : ajoutez 8 indices (identites + adjacences) de votre choix.
# Etape 3 : check() -> solution unique ?

def resoudre_mini_zebra():
    # 1. Définition des groupes
    groups_4 = {
        "Nationalite": ["Francais", "Anglais", "Espagnol", "Italien"],
        "Couleur": ["Rouge", "Bleu", "Vert", "Jaune"],
        "Animal": ["Chien", "Chat", "Poisson", "Oiseau"]
    }

    # Création du solveur
    s = z3.Solver()

    # Création des variables
    v = {
        nom: z3.Int(nom)
        for groupe in groups_4.values()
        for nom in groupe
    }

    # 2. Domaine : chaque variable doit être entre 0 et 3
    for var in v.values():
        s.add(var >= 0, var <= 3)

    # Les 4 valeurs de chaque attribut doivent être dans
    # des maisons différentes
    for groupe in groups_4.values():
        s.add(z3.Distinct([v[nom] for nom in groupe]))

    # 3. Les 8 indices

    # 1. Le Français habite dans la maison rouge
    s.add(v["Francais"] == v["Rouge"])

    # 2. L'Anglais possède le chien
    s.add(v["Anglais"] == v["Chien"])

    # 3. L'Espagnol habite dans la maison verte
    s.add(v["Espagnol"] == v["Vert"])

    # 4. Le Français possède le chat
    s.add(v["Francais"] == v["Chat"])

    # 5. L'Espagnol habite dans la première maison
    s.add(v["Espagnol"] == 0)

    # 6. Le chien est à côté de la maison jaune
    s.add(z3.Or(
        v["Chien"] - v["Jaune"] == 1,
        v["Jaune"] - v["Chien"] == 1
    ))

    # 7. Le poisson est dans la maison verte
    s.add(v["Poisson"] == v["Vert"])

    # 8. La maison bleue est à côté de la maison verte
    s.add(z3.Or(
        v["Bleu"] - v["Vert"] == 1,
        v["Vert"] - v["Bleu"] == 1
    ))

    return s, v


# Test du solveur
s, v = resoudre_mini_zebra()

if s.check() == z3.sat:
    print("Solution trouvée !")

    m = s.model()

    # Affichage des maisons
    for maison in range(4):
        print(f"\nMaison {maison + 1} :")

        for groupe, valeurs in {
            "Nationalité": ["Francais", "Anglais", "Espagnol", "Italien"],
            "Couleur": ["Rouge", "Bleu", "Vert", "Jaune"],
            "Animal": ["Chien", "Chat", "Poisson", "Oiseau"]
        }.items():

            for valeur in valeurs:
                if m[v[valeur]].as_long() == maison:
                    print(f"  {groupe} : {valeur}")

else:
    print("Aucune solution.")

# Etape 3 : solution unique ? On reutilise compter_solutions (exemple guide 2).
s_u, v_u = resoudre_mini_zebra()
print(f"\nNombre de solutions : {compter_solutions(s_u, v_u)} (attendu : 1)")

Solution trouvée !

Maison 1 :
  Nationalité : Espagnol
  Couleur : Vert
  Animal : Poisson

Maison 2 :
  Nationalité : Anglais
  Couleur : Bleu
  Animal : Chien

Maison 3 :
  Nationalité : Italien
  Couleur : Jaune
  Animal : Oiseau

Maison 4 :
  Nationalité : Francais
  Couleur : Rouge
  Animal : Chat

Nombre de solutions : 1 (attendu : 1)


### Lecture du résultat : l'encodage se généralise tel quel

Le même encodage par position passe directement à 4 maisons : domaine `0..3`, 3 groupes `Distinct` et 8 indices (égalités de position et adjacences). Trouver une solution (`sat`) ne suffit pas pour une énigme : il faut aussi qu'elle soit **unique**. En réutilisant `compter_solutions` (exemple guidé 2), on vérifie que les 8 indices déterminent **une seule solution** : l'Espagnol, dans la maison verte, possède le poisson.

### Exercice 3 -- Cinq amies, cinq boissons, cinq loisirs

**Objectif** : modéliser une énigme à **5 maisons** sur un autre thème, avec deux types de relations **nouveaux** par rapport au mini-Zebra : « à gauche quelque part » (`a < b`) et « immédiatement à gauche de » (`a == b - 1`). Cinq amies (Alice, Bruno, Chloe, David, Emilie), cinq boissons (Cafe, The, Jus, Lait, Eau) et cinq loisirs (Echecs, Lecture, Course, Peinture, Musique) — une valeur par maison, une maison par valeur.

Les 11 indices :

1. Alice habite la première maison.
2. Emilie habite la dernière maison.
3. Qui boit du lait fait de la peinture.
4. Bruno boit du thé.
5. Le joueur d'échecs habite quelque part à gauche du buveur de café.
6. David boit du jus.
7. Le coureur habite immédiatement à gauche de Chloe.
8. La lectrice boit de l'eau.
9. Chloe fait de la musique.
10. Le buveur de café habite immédiatement à droite de Bruno.
11. La peinture se pratique dans une maison à gauche de celle de Chloe.

**Indices** : reprenez la structure de `resoudre_mini_zebra` (domaine `0..4`, 3 groupes `Distinct`) ; « immédiatement à gauche de » se code `x == y - 1` et « à gauche quelque part » `x < y`. Vérifiez `sat` **puis** l'unicité avec `compter_solutions(s, v)` — attendu : exactement 1 solution.

In [11]:
# Exercice 3 : cinq amies, cinq boissons, cinq loisirs (5 maisons, 3 attributs, 11 indices).
# Etape 1 : définissez groups_5 (Prenom, Boisson, Loisir), domaine 0..4 + 3 Distinct.
# Etape 2 : encodez les 11 indices de l'énoncé (égalités, adjacences, "à gauche", "immédiatement à gauche").
# Etape 3 : check() -> sat, puis compter_solutions(s, v) -> attendu exactement 1.

def resoudre_cinq_amies():
    """Retourne (solver, var_dict) pour l'énigme des cinq amies (attendu : sat et solution unique)."""
    # TODO etudiant : groups_5 + solveur avec domaine 0..4 + 3 Distinct + les 11 indices
    return None  # TODO etudiant

print("Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).")

Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).


## Conclusion

L'enigme d'Einstein illustre le gain du **declaratif** sur une classe de problemes ou l'approche humaine (deduction progressive sur un tableau) et l'approche naive (brute force) sont toutes deux laborieuses :

- L'**encodage par position** (une variable entiere = position de chaque valeur) transforme un puzzle apparemment qualitatif en un système de contraintes arithmetiques : egalites (`Brit == Red`), adjacences (`|Blend - Cat| == 1`), permutations (`Distinct`).
- La **propagation de contraintes** resout les 25 variables entrelacees en millisecondes la ou la brute force affronte ~25 milliards de combinaisons ; le solveur **prouve l'unicite** en niant le temoin et en observant `unsat`.
- Le modèle est **paramètre par les données** : ajouter un indice, un attribut ou une maison se fait en modifiant l'instance, sans re-ecrire l'algorithme de resolution.

**Contraste avec le job-shop (notebook 08)** : ici `Solver` suffit (on cherche une solution realisable, pas un optimum) - c'est la **satisfiabilite** pure, la ou l'ordonnancement relevait de l'**optimisation** (`Optimize.minimize` du makespan). Les deux postures du solveur, posees au notebook 01, trouvent ici leur illustration complementaire.

Suite logique : retour a la [serie Z3-Python](README.md), ou explorez la version C# [Z3.Linq](../Z3-Linq2Z3/README.md) qui aborde la même enigme via la couche declarative LINQ.